# Interactive Script: **Co-register Data Cube**

**Author:** Baturalp Arisoy<br>
**Contact:** baturalp.arisoy@uni-wuerzburg.de - Call me Batu :)

## Overview
This notebook guides the user through the essentials of co-registration of Sentinel-2 cube. The user can activate: 
1. **coregister_cube** to automatically co-register all available, analysis worth scenes (recommended function). The code ensures maximum amount of possible co-registration by scanning through the entire scene. 
2. **coregister_cube_roi** by manually selecting a polygon using the interactive leafmap by selecting a known stable area on the surface. This function is much faster than first function, however satellite time-series are complicated and can result removing many analysis worth scenes (scenes with very low cloud percentage). Therefore, the first method is always recommended to sustain the maximum amount of available scenes!

> **Note 1:**<br><br>
> This algorithm works the best if the study area is large enough. The larger the area, the better result to get, especially if the surface features are heterogeneous! <br><br>
> In case you are working on small patch but still wish to co-register, please generate initial data cube with much larger area and after co-registration use "from stac2cube import clip_raster" 

> **Note 2:**<br><br>
> The algorithm performs much better if the clouds are masked. <br><br>Even if you want to skip Notebook 2 (Cloud Mask Data Cube generation), please generate your initial data cube (Notebook 1) with SCL masking (cloud_masking = True)

## 1. Auto Co-Registration (Recommended!)

In [1]:
from stac2cube import coregister_cube

In [2]:
out_ds = coregister_cube(
    input_path="./results/coregistered_naryn_sr.nc",          # can be DataArray, Dataset and NetCDF
    grid_size=7, # If the current setup still removes scenes with low cloud percentages, 
                            #try increasing grid_size. It will take longer to process but could result better.
    max_cc=None,
    time_period= None,
    min_reliability_keep=10.0,
    min_reliability_update_ref=70.0,
    max_cloud_update_ref=20.0,
    output_path = None           # If None, coregistered file will be exported to same folder of input, with extra prefix "_cr"
)

Co-registering scenes:   0%|          | 0/40 [00:00<?, ?scene/s]


Co-registration summary
-----------------------
Original (after max_cc/time_period): 40 scenes from 2024-01-25 to 2024-12-25
Scenes excluded after co-registration (overlap / tie points / low reliability): 0
Scenes remaining in the co-registered cube: 40

Mean match reliability of kept scenes: 95.1 %
Minimum match reliability of kept scenes: 87.0 % (date: 2024-04-04)

S2 co-registration is completed!

Co-registered cube written to: ./results\coregistered_naryn_sr_cr.nc


## 2. Custom ROI Co-Registration

In [ ]:
# Select your polygon on the interactive map and continue with the next cell

import leafmap
import numpy as np
import xarray as xr

stac = xr.open_dataset("./results/test.nc")
stac = stac.Spectral_Temporal_Stack

xmin, ymin, xmax, ymax = map(float, np.asarray(stac.bbox))

m = leafmap.Map(height="800px")
m.add_basemap("Google Hybrid")

# leafmap expects bounds as [[south, west], [north, east]] i.e. [[ymin, xmin], [ymax, xmax]]
m.fit_bounds([[ymin, xmin], [ymax, xmax]])

m

In [4]:
from stac2cube import coregister_cube_roi

In [ ]:
# roi (leafmap)
polygon_map = m.user_roi["geometry"]
out_ds = coregister_cube_roi(
    input_path= stac,
    roi= polygon_map,
    max_cc= 5,
    time_period= None, #["2024-04-19", "2024-10-30"]
    min_reliability_keep= 10.0,
    min_reliability_update_ref= 40.0,
    max_cloud_update_ref= 20.0,
    output_path= "./results/roi_coregistration.nc",
)